In [ ]:
import pandas as pd
import yaml
import pdfplumber
from tqdm import tqdm

In [ ]:
import os
os.chdir('../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

In [ ]:
pdf_file = dataset_config['path_cnki'] + '中国学术期刊影响因子年报_自然科学_2021.pdf'
pdf_fp = pdfplumber.open(pdf_file)

PAGE_RANGE = [30, 152]
COLUMNS = [1, 4, 10]

In [ ]:
pp = pdf_fp.pages[30]
im = pp.to_image(resolution=300)
column_cells = pp.find_table(table_settings={}).columns[10].cells
im.draw_cells(column_cells)

In [ ]:
## bbox = (obj["x0"], obj["top"], obj["x1"], obj["bottom"])
column_cells[0]

In [ ]:
def within_cell(word, cell, offset=0.5):
    word_bbox = (word["x0"], word["top"], word["x1"], word["bottom"])
    if (word_bbox[0] + offset > cell[0]) and (word_bbox[2] - offset < cell[2]) and \
       (word_bbox[1] + offset > cell[1]) and (word_bbox[3] - offset < cell[3]):
        return True
    return False

In [ ]:
def extract_cell_words(pp, col_num, x_tol=3, y_tol=3):
    """
    Returns:
      cell_words:   list of lists of word-dicts (one list per cell)
      cell_strings: list of joined text for each cell
    """
    all_words = pp.extract_words(x_tolerance=x_tol, y_tolerance=y_tol)
    cells = pp.find_table(table_settings={}).columns[col_num].cells

    cell_words   = []
    cell_strings = []

    for idx, cell in enumerate(cells):
        if cell is None:
            continue

        curr_words = []

        for word in all_words:
            if within_cell(word, cell):
                curr_words.append(word)
        if not curr_words:
            print(f'## Cell {cell} has NO matches.')
        
        # sort for natural reading order
        curr_words.sort(key=lambda w: (w["top"], w["x0"]))
        cell_words += curr_words

        # join their text into one string
        text = " ".join(w["text"] for w in curr_words)
        cell_strings.append(text)

    return cell_words, cell_strings

In [ ]:
cell_words, cell_strings = extract_cell_words(pp, 10)
len(cell_strings)

In [ ]:
dfs_page = []

for page_num in tqdm(range(PAGE_RANGE[0], PAGE_RANGE[1])):
    pp = pdf_fp.pages[page_num]
    curr_strings = []

    for col_num in COLUMNS:
        cell_words, cell_strings = extract_cell_words(pp, col_num)
        curr_strings.append(cell_strings)
    
    for idx in range(1, len(curr_strings)):
        curr_strings[idx][0] += curr_strings[idx].pop(1).replace(' ', '')
        if len(curr_strings[idx]) - len(curr_strings[0]) != 0:
            print(f'## Page {page_num} column {COLUMNS[idx]} ERROR.')
    
    dfs_page.append(pd.DataFrame(curr_strings).set_index(0).T)

In [ ]:
jif_all = pd.concat(dfs_page).reset_index().drop(columns=['index'])
jif_all

In [ ]:
jif_final = jif_all.drop(jif_all[jif_all['刊名'] == '本栏目计量指标均值'].index)
jif_final

In [ ]:
jif_final.to_csv(dataset_config['path_processed'] + 'CNKI/01_CNKI_JIF_Natural_Science.csv', index=False)

In [ ]:
im = pp.to_image(resolution=300)
im.draw_rects(cell_words)